In [0]:
%sql
-- Notebook: 3_Gold_Analytics
-- Language: SQL


-- 1. Calculate Monthly Average NAV for each fund
-- This table provides a high-level trend view.
CREATE OR REPLACE TABLE mutual_fund_project.gold.monthly_average_nav
AS
SELECT
  scheme_code,
  scheme_name,
  fund_house,
  YEAR(date) as year,
  MONTH(date) as month,
  AVG(nav) as average_nav
FROM mutual_fund_project.silver.fund_nav_details
GROUP BY
  scheme_code, scheme_name, fund_house, YEAR(date), MONTH(date)
ORDER BY
  scheme_code, year, month;

In [0]:
%sql
-- 2. Create a Materialized View for Year-over-Year Performance
-- This calculates daily returns and ranks funds based on their 1-year performance.

CREATE OR REPLACE TABLE mutual_fund_project.gold.fund_performance_yoy AS
WITH DailyReturns AS (
  SELECT
    scheme_code,
    scheme_name,
    date,
    nav,
    LAG(nav, 1) OVER (
      PARTITION BY scheme_code
      ORDER BY date
    ) as nav_previous_day
  FROM mutual_fund_project.silver.fund_nav_details
),
YearlyReturns AS (
  SELECT
    scheme_code,
    scheme_name,
    LAST_VALUE(nav) IGNORE NULLS OVER (
      PARTITION BY scheme_code
      ORDER BY date
      ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) as latest_nav,
    FIRST_VALUE(nav) IGNORE NULLS OVER (
      PARTITION BY scheme_code
      ORDER BY date
      ROWS BETWEEN 365 PRECEDING AND CURRENT ROW
    ) as nav_365_days_ago,
    MAX(date) OVER (
      PARTITION BY scheme_code
    ) as latest_date
  FROM DailyReturns
)
SELECT DISTINCT
  yr.scheme_code,
  yr.scheme_name,
  yr.latest_date,
  yr.latest_nav,
  yr.nav_365_days_ago,
  try_divide(yr.latest_nav - yr.nav_365_days_ago, yr.nav_365_days_ago) * 100 as one_year_return_pct,
  RANK() OVER (
    ORDER BY try_divide(yr.latest_nav - yr.nav_365_days_ago, yr.nav_365_days_ago) DESC
  ) as performance_rank
FROM YearlyReturns yr
WHERE yr.nav_365_days_ago IS NOT NULL;

In [0]:
%sql
select * from mutual_fund_project.gold.fund_performance_yoy limit 10;

# YOY Performance

In [0]:
%sql
-- Year-over-Year Performance
CREATE OR REPLACE TABLE mutual_fund_project.gold.fund_performance_yoy_new
AS
WITH LatestNAV AS (
  -- Find the latest available NAV for each fund
  SELECT
    scheme_code,
    scheme_name,
    date AS latest_date,
    nav AS latest_nav
  FROM (
    SELECT
      scheme_code,
      scheme_name,
      date,
      nav,
      ROW_NUMBER() OVER (PARTITION BY scheme_code ORDER BY date DESC) as rn
    FROM mutual_fund_project.silver.fund_nav_details
  )
  WHERE rn = 1
),
LaggedNAV AS (
  -- Find the NAV from 365 days prior to the latest date
  SELECT
    l.scheme_code,
    l.scheme_name,
    l.latest_date,
    l.latest_nav,
    s.nav AS nav_365_days_ago
  FROM LatestNAV l
  INNER JOIN mutual_fund_project.silver.fund_nav_details AS s
    ON l.scheme_code = s.scheme_code
    AND s.date = DATE_SUB(l.latest_date, 365) -- Find the exact date 365 days ago
)
-- Final aggregation and ranking
SELECT
  ln.scheme_code,
  ln.scheme_name,
  ln.latest_date,
  ln.latest_nav,
  ln.nav_365_days_ago,
  -- Calculate the percentage return over the last year
  ( (ln.latest_nav - ln.nav_365_days_ago) / ln.nav_365_days_ago ) * 100 as one_year_return_pct,
  -- Rank funds based on their 1-year return
  RANK() OVER (ORDER BY ( (ln.latest_nav - ln.nav_365_days_ago) / ln.nav_365_days_ago ) DESC) as performance_rank
FROM LaggedNAV ln
WHERE ln.nav_365_days_ago IS NOT NULL; -- Ensure we have data for the full period

In [0]:
%sql
select * from mutual_fund_project.gold.fund_performance_yoy_new limit 10;

In [0]:
%sql
SELECT
  scheme_code,
  scheme_name,
  latest_date,
  one_year_return_pct,
  performance_rank
FROM mutual_fund_project.gold.fund_performance_yoy_new
WHERE latest_date > '2025-09-01'
ORDER BY
  performance_rank;

# Sharpe Ratio

In [0]:
%sql
CREATE OR REPLACE VIEW mutual_fund_project.gold.gold_mv_sharpe_ratio AS
WITH DailyReturns AS (
  SELECT
    scheme_code,
    scheme_name,
    date,
    try_divide(
      nav - LAG(nav, 1) OVER (
        PARTITION BY scheme_code
        ORDER BY date
      ),
      LAG(nav, 1) OVER (
        PARTITION BY scheme_code
        ORDER BY date
      )
    ) AS daily_return_pct
  FROM mutual_fund_project.silver.fund_nav_details
),
AggregatedMetrics AS (
  SELECT
    scheme_code,
    scheme_name,
    AVG(daily_return_pct) AS avg_daily_return,
    STDDEV_SAMP(daily_return_pct) AS stddev_daily_return
  FROM DailyReturns
  WHERE date >= DATE_SUB(CURRENT_DATE(), 365)
  GROUP BY
    scheme_code, scheme_name
)
SELECT
  am.scheme_code,
  am.scheme_name,
  (POWER(1 + am.avg_daily_return, 365) - 1) AS annualized_return,
  am.stddev_daily_return * POWER(365, 0.5) AS annualized_stddev,
  try_divide(
    (POWER(1 + am.avg_daily_return, 365) - 1) - 0.07,
    am.stddev_daily_return * POWER(365, 0.5)
  ) AS sharpe_ratio
FROM AggregatedMetrics am
WHERE am.stddev_daily_return IS NOT NULL;

In [0]:
%sql
select * from mutual_fund_project.gold.gold_mv_sharpe_ratio